In [5]:
import warnings
warnings.filterwarnings('ignore')

import os
import math
import pandas as pd
import numpy as np
import joblib
import pickle
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

In [2]:
def set_wd():
    # Get the GitHub Actions workspace directory
    workspace = os.getenv('GITHUB_WORKSPACE', '.')
    
    # Set the working directory to the folder where the data resides
    cleaned_data = os.path.join(workspace, 'cleaned data')
    website_code = os.path.join(workspace, 'Website code')
    return cleaned_data,website_code

In [3]:
def round_decimals_up(number:float, decimals:int=2):
    """
    Returns a value rounded up to a specific number of decimal places.
    """
    if not isinstance(decimals, int):
        raise TypeError("decimal places must be an integer")
    elif decimals < 0:
        raise ValueError("decimal places has to be 0 or more")
    elif decimals == 0:
        return math.ceil(number)

    factor = 10 ** decimals
    return math.ceil(number * factor) / factor

In [4]:
def load_data():
    os.chdir(cleaned_data)
    match_results = pd.read_csv('afl_match_results_cleaned.csv')
    team_stats = pd.read_csv('afl_team_stats_cleaned.csv')
    win_streaks = pd.read_csv('afl_team_streaks_cleaned.csv',index_col=0)
    venue_streaks = pd.read_csv('afl_venue_streaks_cleaned.csv',index_col=0)
    team_form = pd.read_csv('afl_team_form_cleaned.csv',index_col=0)
    fixture = pd.read_csv('afl_fixture_cleaned.csv',index_col=0)
    fixture = fixture[fixture['home.team.name'].isin(pd.unique(team_stats['Team'].values.ravel('K'))) & fixture['away.team.name'].isin(pd.unique(team_stats['Team'].values.ravel('K')))]
    os.chdir(website_code)
    return match_results,team_stats,win_streaks,venue_streaks,team_form,fixture

In [5]:
def extract_features(home_team, away_team, venue,weather):
    # Get the weighted average stats for home and away teams
    home_stats = team_stats[team_stats['Team'] == home_team].iloc[:, 1:].values.flatten()
    away_stats = team_stats[team_stats['Team'] == away_team].iloc[:, 1:].values.flatten()
    
    if venue in team_form.columns:
        home_venue_streak = venue_streaks.loc[home_team, venue].flatten()
        away_venue_streak = venue_streaks.loc[away_team, venue].flatten()
    else:
        home_venue_streak = 0
        away_venue_streak = 0
    
    # Get the win streaks
    team_win_streak = win_streaks.loc[away_team, home_team].flatten()
    home_team_form = team_form.loc[team_form['Team'] == home_team, 'Current.Form'].values[0].flatten()
    away_team_form = team_form.loc[team_form['Team'] == home_team, 'Current.Form'].values[0].flatten()
    
    # Combine all features into a single array
    features = np.concatenate([
    home_stats, home_team_form,
    away_stats, away_team_form,
    home_venue_streak,away_venue_streak,team_win_streak,
    ])

    df1 = pd.DataFrame([features])
    df2 = pd.DataFrame([weather])
    features=pd.concat([df1, df2], axis = 1)

    column_names = match_results.drop(columns=['match.homeTeam.name', 'match.awayTeam.name','venue.name','Margin','Result','weather.weatherType',
                                          'Home.Team.Venue.Win.Streak', 'Away.Team.Venue.Win.Streak','Home.Win.Streak']).columns  # Replace with actual feature names
    column_names = column_names.append(pd.Index(['Home.Team.Venue.Win.Streak', 'Away.Team.Venue.Win.Streak','Home.Win.Streak'])).append(pd.Index(['weather.weatherType']))
    
    features.columns = column_names
    
    return features

In [12]:
def make_prediction(home_team, away_team, venue,weather):
    with open('accuracy.pkl', 'rb') as f:
        a = pickle.load(f)
    features = extract_features(home_team, away_team, venue,weather)
    pred_probs = model.predict_proba(features)  # Get the probability for each class
    pred_class = np.argmax(pred_probs, axis=1)  # Class with the highest probability
    pred_class = encoder.inverse_transform([pred_class])[0]
    predicted_prob = np.max(pred_probs, axis=1)
    acc = predicted_prob[0] * a
    max_prob_percent = f"{acc * 100:.2f}%"
    market = f"{round_decimals_up(1 / acc,2):.2f}"
    
    return pred_class,max_prob_percent,market

In [7]:
def gen_predictions():
    preds = []
    for h,a,v,w in zip(list(fixture['home.team.name']),
                                             list(fixture['away.team.name']),
                                             list(fixture['venue.name']),
                                             list(fixture['Next_round_weather'])):
        (r,p,m)=make_prediction(h,a,v,w)

        if r == "BW":
            r = f'{h} 40+'
        if r == "LW":
            r = f'{h} 1-39'
        if r == "LL":
            r = f'{a} 1-39'
        if r == "BL":
            r = f'{a} 40+'
        if r == "D":
            r = f'draw'
    
        new = {"Match": f"{h} vs {a}", "Venue": f"{v}" , "Prediction": f"{r}",
             "Market": f"${m}", "Probability": f"{p}"}
        preds.append(new)
    return preds

In [8]:
def output_predictions_json():
    # Prepare data for JSON
    output_data = p
    
    os.chdir(website_code)
    # Save to JSON
    with open(f'predictions.json', 'w') as f:
        json.dump(output_data, f, indent=4)
    print(f'Predictions saved to predictions.json')

In [15]:
if __name__ == '__main__':
    with open('encoder.pkl', 'rb') as f:
        encoder = pickle.load(f)
    with open('preprocessor.pkl', 'rb') as f:
        preprocessor = pickle.load(f)
    model = joblib.load('knn_model.pkl')

    cleaned_data,website_code=set_wd()
    weather_categories = ['CLEAR_NIGHT','MOSTLY_SUNNY','OVERCAST','RAIN','SUNNY','THUNDERSTORMS','WINDY']  # Add all weather types you used
    # Create a dictionary where all categories are 0
    weather_dict = {category: 0 for category in weather_categories}
    
    match_results, team_stats, win_streaks, venue_streaks, team_form, fixture = load_data()

    p = gen_predictions()
    output_predictions_json()

FileNotFoundError: [WinError 2] The system cannot find the file specified: '.\\cleaned data'

In [19]:
team_form = pd.read_csv('C:\\Users\\blake\\Desktop\\AFL Odds\\cleaned data\\afl_venue_streaks_cleaned.csv',index_col=0)

In [21]:
team_form.columns

Index(['Accor Stadium', 'Adelaide Arena at Jiangwan Stadium', 'Adelaide Hills',
       'Adelaide Oval', 'Blacktown ISP', 'Cazalys Stadium', 'ENGIE Stadium',
       'Football Park', 'Gabba', 'GMHBA Stadium', 'M.C.G.', 'Manuka Oval',
       'Mars Stadium', 'Marvel', 'Ninja Stadium', 'Norwood Oval',
       'Optus Stadium', 'People First Stadium', 'Riverway Stadium', 'S.C.G.',
       'Subiaco', 'TIO Stadium', 'TIO Traeger Park', 'UTAS Stadium',
       'Wellington'],
      dtype='object')

In [25]:
venue = 'Barossa Park'
if venue in team_form.columns:
    print("Y")
else: print("N")

N
